# Opening files and Data Cleaning

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt

## Stops
Locations of all public transport stops (train and bus) in General Transit Feed Specification (GTFS) format.

In [4]:
stops = pd.read_csv("Stops.txt", delimiter=",", engine='python')
print(stops.shape)
stops.head(5)

(114718, 9)


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
0,200039,200039.0,"Central Station, Eddy Av, Stand A",-33.882206,151.206665,NaN,200060,0,NaN
1,200054,200054.0,"Central Station, Eddy Av, Stand D",-33.882042,151.206991,NaN,200060,0,NaN
2,200060,NaN,Central Station,-33.884084,151.206292,1.0,NaN,0,NaN
3,201510,NaN,Redfern Station,-33.891690,151.198866,1.0,NaN,0,NaN
4,201646,201646.0,"Redfern Station, Gibbons St, Stand B",-33.893329,151.198882,NaN,201510,0,NaN


In [44]:
stops.describe()

,stop_code,stop_lat,stop_lon,location_type,wheelchair_boarding
count,6.071900e+04,114718.000000,114718.000000,53991.0,114718.000000
mean,1.943536e+06,-33.411399,150.653190,1.0,0.011942
std,3.732023e+06,1.462086,1.522872,0.0,0.126566
min,1.300000e+02,-37.817644,138.588356,1.0,0.000000
25%,2.317125e+05,-33.961518,150.613331,1.0,0.000000
50%,2.142172e+06,-33.755680,150.992310,1.0,0.000000
75%,2.460240e+06,-32.997889,151.295420,1.0,0.000000
max,2.830112e+07,-27.464295,153.619512,1.0,2.000000


In [29]:
stops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114718 entries, 0 to 114717
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   stop_id              114718 non-null  object 
 1   stop_code            60719 non-null   float64
 2   stop_name            114718 non-null  object 
 3   stop_lat             114718 non-null  float64
 4   stop_lon             114718 non-null  float64
 5   location_type        53991 non-null   float64
 6   parent_station       60727 non-null   object 
 7   wheelchair_boarding  114718 non-null  int64  
 8   platform_code        871 non-null     object 
dtypes: float64(4), int64(1), object(4)
memory usage: 7.9+ MB


### Null values

In [30]:
print("Number of null values:")
for column, null_count in stops.isnull().sum().items():
    if null_count != 0:
        print(f"{column}:   {null_count},    {int(null_count/len(stops)*100)}%")
if stops.isnull().sum().sum() == 0:
    print("Nice! There are no missing values.")


Number of null values:
stop_code:   53999,    47%
location_type:   60727,    52%
parent_station:   53991,    47%
platform_code:   113847,    99%


### No duplicate rows

In [16]:
duplicate_rows = stops.duplicated()
if duplicate_rows.any():
    print("There are duplicate rows in the DataFrame.")
else:
    print("There are no duplicate rows in the DataFrame.")

There are duplicate rows in the DataFrame.


### No duplicate stop id

In [34]:
column_name = 'stop_id'

duplicate_values_in_column = stops.duplicated(subset=[column_name])

if duplicate_values_in_column.any():
    print(f"There are duplicate values in the column '{column_name}'.")
else:
    print(f"There are no duplicate values in the column '{column_name}'.")


There are no duplicate values in the column 'stop_id'.


### Stop_id == stop_code ?

In [12]:
stops['stop_id'] = pd.to_numeric(stops['stop_id'], errors='coerce')
stops.loc[stops['stop_id'] == (stops['stop_code'])]

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
0,200039.0,200039.0,"Central Station, Eddy Av, Stand A",-33.882206,151.206665,NaN,200060,0,NaN
1,200054.0,200054.0,"Central Station, Eddy Av, Stand D",-33.882042,151.206991,NaN,200060,0,NaN
4,201646.0,201646.0,"Redfern Station, Gibbons St, Stand B",-33.893329,151.198882,NaN,201510,0,NaN
5,204230.0,204230.0,"St Peters Station, King St",-33.906314,151.181117,NaN,204410,0,NaN
6,204311.0,204311.0,King St Opp St Peters Station,-33.906423,151.181371,NaN,204410,0,NaN
...,...,...,...,...,...,...,...,...,...
114712,212751.0,212751.0,"Sydney Olympic Park Wharf, Side A",-33.821961,151.078827,NaN,21271,1,A
114713,212753.0,212753.0,"Sydney Olympic Park Wharf, Side B",-33.822016,151.078797,NaN,21271,1,B
114714,2137185.0,2137185.0,"Cabarita Wharf, Side A",-33.840669,151.116926,NaN,21371,1,1A
114715,2137186.0,2137186.0,"Cabarita Wharf, Side B",-33.840769,151.116899,NaN,21371,1,1B


In [18]:
53999 + 60719, len(stops)

(114718, 114718)

### Duplicate stop names

In [19]:
column_name = 'stop_name'

duplicate_values_in_column = stops.duplicated(subset=[column_name])

if duplicate_values_in_column.any():
    print(f"There are duplicate values in the column '{column_name}'.")
else:
    print(f"There are no duplicate values in the column '{column_name}'.")

There are duplicate values in the column 'stop_name'.


### Wheelchair boarding in numbers

In [21]:
occurrences = stops['wheelchair_boarding'].value_counts()
print(occurrences)


0    113590
1       886
2       242
Name: wheelchair_boarding, dtype: int64


### Platform code

In [25]:
stops.loc[stops['platform_code'].notnull()]

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
268,2000321.0,2000321.0,"Central Station, Platform 1",-33.882703,151.205857,NaN,200060,1,1
269,2000322.0,2000322.0,"Central Station, Platform 2",-33.882751,151.206011,NaN,200060,1,2
270,2000323.0,2000323.0,"Central Station, Platform 3",-33.882768,151.206066,NaN,200060,1,3
271,2000324.0,2000324.0,"Central Station, Platform 4",-33.883325,151.205834,NaN,200060,1,4
272,2000325.0,2000325.0,"Central Station, Platform 5",-33.883351,151.205878,NaN,200060,1,5
...,...,...,...,...,...,...,...,...,...
114705,211033.0,211033.0,"Cockatoo Island Wharf, Side A",-33.845513,151.173472,NaN,20009,1,A
114712,212751.0,212751.0,"Sydney Olympic Park Wharf, Side A",-33.821961,151.078827,NaN,21271,1,A
114713,212753.0,212753.0,"Sydney Olympic Park Wharf, Side B",-33.822016,151.078797,NaN,21271,1,B
114714,2137185.0,2137185.0,"Cabarita Wharf, Side A",-33.840669,151.116926,NaN,21371,1,1A


#### Observations:

* It looks like parent stations such as: Central, Redfern and St Peters, have always NaN values for the stop_code and the parent_station columns, instead they are the only one having a non NaN value in the location_type column which appears to be always 1.0. 

* stop_id and stop_code ofte refer to the same number, but stop_code has many null values (but as pointed above this corresponds to stations being parent stations or not)

* the other column with many null values, 'platform_code' has nonull values only when there are multiple platforms in a station which makes sense

So overall, the dataset is well structured, and the null values don't seem to represent any mistake.² 

#### Suggestions:
* First, when converting the dataframe to an SQL table the best candidate for the primary key is stop_id. Since it has no null values, neither duplicate values.
* We could remove the stop_code column since it is essentialy a copy of the stop_id column, but just with some null values. Nevertheless, we noticed that those null values in the stop_code correspond to parent stations, so we might might consider keeping them.

Finally, we should consider what attributes might be useful for our 'bustling measure' and which are not. However, in this early data cleaning phase we will try to keep as much data as possible and address the aspect in following steps.